In [ ]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os


# Get all SAS files in the data folder
data_folder = '../data/'
sas_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.sas7bdat')])

print(f"Found {len(sas_files)} SAS files:")
for file in sas_files:
    print(f"  - {file}")

# Load all SAS files and combine them
dfs = []
for file in sas_files:
    file_path = os.path.join(data_folder, file)
    print(f"\nLoading {file}...")
    df_temp = pd.read_sas(file_path)
    print(f"  Shape: {df_temp.shape}")
    print(f"  Columns: {list(df_temp.columns)}")
    dfs.append(df_temp)

# Combine all dataframes
df = pd.concat(dfs, ignore_index=True)

print(f"\n{'='*60}")
print(f"COMBINED DATASET")
print(f"{'='*60}")
print(f"Total Shape: {df.shape}")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 3-4: truncated \UXXXXXXXX escape (1812739508.py, line 9)

In [27]:
# Dataset Overview
print("Dataset Shape:", df.shape)
print("\nColumn Names and Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (109760, 1061)

Column Names and Data Types:
VMONTH              float64
VDAYR               float64
ARRTIME              object
WAITTIME            float64
LOV                 float64
                     ...   
BLANK3              float64
BLANK4              float64
COVID_VALIDATION    float64
COVIDTEST           float64
COVIDANTIBODY       float64
Length: 1061, dtype: object

Missing Values:
VMONTH                   0
VDAYR                    0
ARRTIME                  0
WAITTIME                 0
LOV                  36176
                     ...  
BLANK3              109760
BLANK4              109760
COVID_VALIDATION     93553
COVIDTEST            93735
COVIDANTIBODY        93735
Length: 1061, dtype: int64


In [28]:
# Convert ARRTIME to proper time format (HH:MM)
def convert_arrtime_to_time(arr_time):
    """
    Convert ARRTIME from byte format (e.g., b'0604') to time format (HH:MM)
    b'0604' -> '06:04'
    b'1419' -> '14:19'
    """
    try:
        if pd.isna(arr_time):
            return pd.NaT
        
        # Convert bytes to string if needed
        if isinstance(arr_time, bytes):
            time_str = arr_time.decode('utf-8')
        else:
            time_str = str(arr_time)
        
        # Handle cases where time is already in correct format or is missing
        if len(time_str) < 4:
            return pd.NaT
        
        # Extract hours and minutes (last 4 digits)
        hours = time_str[-4:-2]
        minutes = time_str[-2:]
        
        # Create time string in HH:MM format
        time_formatted = f"{hours}:{minutes}"
        
        # Convert to datetime.time object
        return pd.to_datetime(time_formatted, format='%H:%M').time()
    except:
        return pd.NaT

# Apply conversion
df['ARRTIME'] = df['ARRTIME'].apply(convert_arrtime_to_time)

print("ARRTIME conversion complete!")
print("\nSample converted ARRTIME values:")
print(df['ARRTIME'].head(20))
print(f"\nData type: {df['ARRTIME'].dtype}")
print(f"\nMissing values: {df['ARRTIME'].isna().sum()}")



ARRTIME conversion complete!

Sample converted ARRTIME values:
0     12:36:00
1     21:14:00
2     16:19:00
3     09:50:00
4     17:38:00
5     16:53:00
6     16:32:00
7     14:04:00
8     16:01:00
9     12:42:00
10    21:31:00
11    16:56:00
12    15:39:00
13    13:49:00
14    22:18:00
15    11:55:00
16    20:56:00
17    21:50:00
18    18:20:00
19    15:01:00
Name: ARRTIME, dtype: object

Data type: object

Missing values: 1989


In [ ]:
# Extract column descriptions from NHAMCS documentation PDF
import pdfplumber
import pandas as pd
import re

pdf_path = '../data/doc21-ed-508.pdf'

# Extract text and tables from PDF
try:
    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages in PDF: {len(pdf.pages)}\n")
        
        # Extract all text
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text()
        
        # Try to find column/variable definitions
        print("Looking for column definitions in the documentation...\n")
        
        # Search for common patterns in variable documentation
        lines = full_text.split('\n')
        
        # Find variable definitions (usually start with a variable name followed by description)
        variables = {}
        current_var = None
        
        for i, line in enumerate(lines):
            # Look for lines that might contain variable definitions
            if re.match(r'^[A-Z]{2,}', line.strip()) and len(line.strip()) < 50:
                current_var = line.strip()
                if i + 1 < len(lines):
                    variables[current_var] = lines[i + 1].strip()
        
        print(f"Found {len(variables)} potential variables")
        print("\nSample variables and descriptions:")
        for var, desc in list(variables.items())[:20]:
            print(f"{var}: {desc[:70]}...")
        
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Total pages in PDF: 266



In [ ]:
# Build a comprehensive data dictionary from PDF and current dataframe columns
column_descriptions = {
    # Temporal Variables
    'VMONTH': 'Month of visit',
    'VDAYR': 'Day of week of visit',
    'ARRTIME': 'Arrival time (formatted)',
    'WAITTIME': 'Time from arrival to seeing provider (minutes)',
    'LOV': 'Length of visit (minutes)',
    'BOARDED': 'Number of hours patient remained in ED after decision to admit',
    
    # Demographics
    'AGE': 'Age of patient in years',
    'AGER': 'Age group',
    'AGEDAYS': 'Age in days for pediatric patients',
    'SEX': 'Sex of patient (1=Male, 2=Female)',
    'ETHUN': 'Ethnicity - unimputed (1=Non-Hispanic, 2=Hispanic)',
    'RACEUN': 'Race - unimputed',
    'RESIDNCE': 'Residence area (1=Metropolitan, 2=Micropolitan, 3=Non-core)',
    
    # Clinical Variables
    'TRIAGE': 'Triage acuity level',
    'PAINSCALE': 'Pain scale score (0-10)',
    'TEMPF': 'Temperature in Fahrenheit',
    'PULSE': 'Heart rate (beats per minute)',
    'RESPR': 'Respiratory rate per minute',
    'BPSYS': 'Blood pressure - Systolic',
    'BPDIAS': 'Blood pressure - Diastolic',
    'POPCT': 'Pulse oximetry (percent)',
    
    # Resource Utilization
    'IMAG': 'Imaging ordered',
    'CT': 'CT scan ordered',
    'MRI': 'MRI ordered',
    'XRAY': 'X-ray ordered',
    'ULTRASOUND': 'Ultrasound ordered',
    'LABTEST': 'Laboratory test ordered',
    'MEDORTO': 'Medication ordered/provided at ED',
    
    # Hospital Operations
    'SETTYPE': 'Type of facility (hospital/freestanding)',
    'YEAR': 'Year of data collection',
    'EDWT': 'Probability weight for ED',
    'PATWT': 'Patient sampling weight',
}

# Get all columns from the dataframe
all_columns = df.columns.tolist()

# Create a DataFrame with column information
column_info = []
for col in all_columns:
    description = column_descriptions.get(col, 'See NHAMCS documentation')
    data_type = str(df[col].dtype)
    non_missing = df[col].notna().sum()
    missing_pct = (df[col].isna().sum() / len(df)) * 100
    
    column_info.append({
        'Column_Name': col,
        'Data_Type': data_type,
        'Non_Missing': non_missing,
        'Missing_%': f"{missing_pct:.1f}%",
        'Description': description
    })

column_dict_df = pd.DataFrame(column_info)

print("="*100)
print("COMPLETE DATA DICTIONARY - ED 2022 DATASET (from NHAMCS)")
print("="*100)
print(f"\nTotal columns: {len(column_dict_df)}")
print(f"Total rows: {len(df)}\n")

# Display by category
print("\n" + "="*100)
print("TEMPORAL VARIABLES")
print("="*100)
temporal_display = column_dict_df[column_dict_df['Column_Name'].isin(['VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV', 'BOARDED'])]
print(temporal_display.to_string(index=False))

print("\n" + "="*100)
print("DEMOGRAPHIC VARIABLES")
print("="*100)
demo_display = column_dict_df[column_dict_df['Column_Name'].isin(['AGE', 'AGER', 'AGEDAYS', 'SEX', 'ETHUN', 'RACEUN', 'RESIDNCE'])]
print(demo_display.to_string(index=False))

print("\n" + "="*100)
print("CLINICAL VARIABLES")
print("="*100)
clinical_display = column_dict_df[column_dict_df['Column_Name'].isin(['TRIAGE', 'PAINSCALE', 'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT'])]
print(clinical_display.to_string(index=False))

print("\n" + "="*100)
print("ALL COLUMNS - FULL DATA DICTIONARY")
print("="*100)
print(column_dict_df.to_string(index=False))

COMPLETE DATA DICTIONARY - ED 2022 DATASET (from NHAMCS)

Total columns: 1062
Total rows: 109760


TEMPORAL VARIABLES
Column_Name Data_Type  Non_Missing Missing_%                                                    Description
     VMONTH   float64       109760      0.0%                                                 Month of visit
      VDAYR   float64       109760      0.0%                                           Day of week of visit
    ARRTIME    object       107771      1.8%                                       Arrival time (formatted)
   WAITTIME   float64       109760      0.0%                 Time from arrival to seeing provider (minutes)
        LOV   float64        73584     33.0%                                      Length of visit (minutes)
    BOARDED   float64        73584     33.0% Number of hours patient remained in ED after decision to admit

DEMOGRAPHIC VARIABLES
Column_Name Data_Type  Non_Missing Missing_%                                                 Descriptio

In [ ]:
# Split columns into thematic dataframes for analysis
from collections import OrderedDict

# Define core groups (explicit + keyword-based fallbacks)
groups = OrderedDict({
    "temporal": {
        "explicit": ["VMONTH", "VDAYR", "ARRTIME", "WAITTIME", "LOV", "BOARDED"],
        "keywords": ["time", "month", "day", "arrival", "wait", "los", "lov", "board", "hour", "minute"]
    },
    "demographics": {
        "explicit": ["AGE", "AGER", "AGEDAYS", "SEX", "ETHUN", "RACEUN", "RESIDNCE"],
        "keywords": ["age", "sex", "gender", "race", "ethnicity", "ethn", "resid", "payer", "insur"]
    },
    "clinical": {
        "explicit": ["TRIAGE", "PAINSCALE", "TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT"],
        "keywords": ["triage", "pain", "vital", "temp", "pulse", "resp", "bp", "ox", "diagn", "injur"]
    },
    "resources": {
        "explicit": ["IMAG", "CT", "MRI", "XRAY", "ULTRASOUND", "LABTEST", "MEDORTO"],
        "keywords": ["imag", "ct", "mri", "xray", "ultra", "lab", "test", "proc"]
    },
    "operations": {
        "explicit": ["SETTYPE", "YEAR", "EDWT", "PATWT"],
        "keywords": ["settype", "edwt", "patwt", "weight", "volume", "bed", "capacity", "fast", "track", "protocol"]
    },
})

# Helper to collect existing columns

def collect_columns(explicit, keywords):
    cols = []
    for c in explicit:
        if c in df.columns:
            cols.append(c)
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in keywords):
            cols.append(c)
    seen = set()
    unique_cols = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            unique_cols.append(c)
    return unique_cols

# Build dataframes
df_temporal = df[collect_columns(groups["temporal"]["explicit"], groups["temporal"]["keywords"])].copy()
df_demographics = df[collect_columns(groups["demographics"]["explicit"], groups["demographics"]["keywords"])].copy()
df_clinical = df[collect_columns(groups["clinical"]["explicit"], groups["clinical"]["keywords"])].copy()
df_resources = df[collect_columns(groups["resources"]["explicit"], groups["resources"]["keywords"])].copy()
df_operations = df[collect_columns(groups["operations"]["explicit"], groups["operations"]["keywords"])].copy()

# Summary
summary = pd.DataFrame({
    "dataframe": ["df_temporal", "df_demographics", "df_clinical", "df_resources", "df_operations"],
    "rows": [len(df_temporal), len(df_demographics), len(df_clinical), len(df_resources), len(df_operations)],
    "columns": [df_temporal.shape[1], df_demographics.shape[1], df_clinical.shape[1], df_resources.shape[1], df_operations.shape[1]]
})

print("Created thematic dataframes:")
print(summary.to_string(index=False))

# Show a few columns from each for validation
print("\nSample columns:")
print("Temporal:", list(df_temporal.columns[:12]))
print("Demographics:", list(df_demographics.columns[:12]))
print("Clinical:", list(df_clinical.columns[:12]))
print("Resources:", list(df_resources.columns[:12]))
print("Operations:", list(df_operations.columns[:12]))




Created thematic dataframes:
      dataframe   rows  columns
    df_temporal 109760       11
df_demographics 109760       18
    df_clinical 109760       21
   df_resources 109760       26
  df_operations 109760        9

Sample columns:
Temporal: ['VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV', 'BOARDED', 'AGEDAYS', 'LOS', 'BOARD', 'BOARDHOS', 'SURGDAY']
Demographics: ['AGE', 'AGER', 'AGEDAYS', 'SEX', 'ETHUN', 'RACEUN', 'RESIDNCE', 'RACER', 'RACERETH', 'ANYIMAGE', 'OTHIMAGE', 'AGEFL']
Clinical: ['PAINSCALE', 'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'INJURY', 'INJURY72', 'TOXSCREN', 'BPAP', 'VITALSD']
Resources: ['MRI', 'XRAY', 'POPCT', 'ELECTROL', 'LACTATE', 'HIVTEST', 'FLUTEST', 'PREGTEST', 'OTHRTEST', 'ANYIMAGE', 'CTCONTRAST', 'CTAB']
Operations: ['SETTYPE', 'YEAR', 'EDWT', 'PATWT', 'BEDREG', 'IMBED', 'FASTTRAK', 'BEDCZAR', 'BEDDATA']


In [ ]:
df_temporal.info()
df_demographics.info()
df_clinical.info()
df_resources.info()
df_operations.info()

print("first 5 rows of temporal:")
print(df_temporal.head())

print("first 5 rows of demographics:")
print(df_demographics.head())       

print("first 5 rows of clinical:")
print(df_clinical.head())

print("first 5 rows of resources:")
print(df_resources.head())  

print("first 5 rows of operations:")
print(df_operations.head()) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109760 entries, 0 to 109759
Data columns (total 11 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   VMONTH    109760 non-null  float64
 1   VDAYR     109760 non-null  float64
 2   ARRTIME   107771 non-null  object 
 3   WAITTIME  109760 non-null  float64
 4   LOV       73584 non-null   float64
 5   BOARDED   73584 non-null   float64
 6   AGEDAYS   109760 non-null  float64
 7   LOS       73584 non-null   float64
 8   BOARD     109760 non-null  float64
 9   BOARDHOS  109760 non-null  float64
 10  SURGDAY   89469 non-null   float64
dtypes: float64(10), object(1)
memory usage: 9.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109760 entries, 0 to 109759
Data columns (total 18 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   AGE       109760 non-null  float64
 1   AGER      109760 non-null  float64
 2   AGEDAYS   109760 non-null  float64
 3   

In [ ]:
# Drop AGEDAYS from df_temporal
df_temporal = df_temporal.drop(columns=['AGEDAYS'], errors='ignore')


Found 31 drug-related columns:
['DRUGID1', 'DRUGID2', 'DRUGID3', 'DRUGID4', 'DRUGID5', 'DRUGID6', 'DRUGID7', 'DRUGID8', 'DRUGID9', 'DRUGID10'] ...

Added 'drug' column to df_resources
Non-null drug entries: 86388

Sample drug entries:
0    DRUGID1:b'd03431'; DRUGID2:b'd00015'; drug:DRU...
1    DRUGID1:b'd05781'; DRUGID2:b'd00059'; DRUGID3:...
2    DRUGID1:b'd00015'; DRUGID2:b'd00965'; drug:DRU...
3    DRUGID1:b'd00015'; drug:DRUGID1:b'd00015'; dru...
4    DRUGID1:b'd00960'; DRUGID2:b'd00212'; DRUGID3:...
5    DRUGID1:b'd00043'; drug:DRUGID1:b'd00043'; dru...
6    DRUGID1:b'd00350'; drug:DRUGID1:b'd00350'; dru...
7    DRUGID1:b'd00015'; drug:DRUGID1:b'd00015'; dru...
8    DRUGID1:b'd00749'; DRUGID2:b'd03827'; drug:DRU...
9    DRUGID1:b'd00046'; DRUGID2:b'd05781'; drug:DRU...
Name: drug, dtype: object

df_temporal shape after dropping AGEDAYS: (109760, 10)
df_resources shape with new drug column: (109760, 125)


,MRI,XRAY,POPCT,IMMEDR,ELECTROL,LACTATE,HIVTEST,FLUTEST,PREGTEST,OTHRTEST,...,DRUGID23,DRUGID24,DRUGID25,DRUGID26,DRUGID27,DRUGID28,DRUGID29,DRUGID30,COVIDTEST,drug
0,0.0,1.0,97.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DRUGID1:b'd03431'; DRUGID2:b'd00015'; drug:DRU...
1,0.0,1.0,96.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DRUGID1:b'd05781'; DRUGID2:b'd00059'; DRUGID3:...
2,0.0,1.0,99.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DRUGID1:b'd00015'; DRUGID2:b'd00965'; drug:DRU...
3,0.0,1.0,100.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DRUGID1:b'd00015'; drug:DRUGID1:b'd00015'; dru...
4,0.0,0.0,99.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DRUGID1:b'd00960'; DRUGID2:b'd00212'; DRUGID3:...


In [ ]:
# Combining all columns from the thematic dataframes to create a comprehensive dataframe for analysis
df_all = pd.concat([df_temporal, df_demographics, df_clinical, df_resources, df_operations], axis=1)
df_all.info()
df_all.drop(columns=['MUSTAGE1', 'MUSTAGE2', ], errors='ignore', inplace=True)
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109760 entries, 0 to 109759
Data columns (total 85 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   VMONTH        109760 non-null  float64
 1   VDAYR         109760 non-null  float64
 2   ARRTIME       107771 non-null  object 
 3   WAITTIME      109760 non-null  float64
 4   LOV           73584 non-null   float64
 5   BOARDED       73584 non-null   float64
 6   AGEDAYS       109760 non-null  float64
 7   LOS           73584 non-null   float64
 8   BOARD         109760 non-null  float64
 9   BOARDHOS      109760 non-null  float64
 10  SURGDAY       89469 non-null   float64
 11  AGE           109760 non-null  float64
 12  AGER          109760 non-null  float64
 13  AGEDAYS       109760 non-null  float64
 14  SEX           109760 non-null  float64
 15  ETHUN         109760 non-null  float64
 16  RACEUN        109760 non-null  float64
 17  RESIDNCE      109760 non-null  float64
 18  RACE